In [5]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

import inspect
import html
from IPython.display import display, HTML


PROJECT_SRC = Path.cwd().parents[1]   # from tests/notebooks -> python/src
if str(PROJECT_SRC) not in sys.path:
    sys.path.insert(0, str(PROJECT_SRC))

from tests.ex1_lqr_test import (
    build_lqr_example_1,
    riccati_rhs,
    solve_riccati_equation,
    riccati_diagnostics,
    state_rhs,
    solve_state_equation,
    compute_lqr_cost,
    pmp_diagnostics,
    run_lqr_riccati_benchmark,
    plot_lqr_trajectories,
)

# Optional (recommended) for syntax highlighting
from pygments import highlight
from pygments.lexers import PythonLexer
from pygments.formatters import HtmlFormatter

_PYGMENTS_CSS_LOADED = False

def show_full_function(func, name=None):
    global _PYGMENTS_CSS_LOADED
    
    name = name or func.__name__
    source = inspect.getsource(func)

    formatter = HtmlFormatter(style="friendly", noclasses=False)
    highlighted = highlight(source, PythonLexer(), formatter)

    if not _PYGMENTS_CSS_LOADED:
        css = formatter.get_style_defs('.highlight')
        display(HTML(f"<style>{css}</style>"))
        _PYGMENTS_CSS_LOADED = True

    html_block = f"""
    <details>
      <summary><strong>View full function: {html.escape(name)}</strong></summary>
      {highlighted}
    </details>
    """
    display(HTML(html_block))

# Example 1: Linear-Quadratic Regulator (LQR)

We consider the finite-horizon optimal control problem
\begin{equation}
\min_{u(\cdot)} J[u]
=
x(T)^\top Q_T x(T)
+
\int_0^T \left( x(t)^\top Q x(t) + u(t)^\top R u(t) \right)\,dt,
\end{equation}
subject to the linear dynamics
\begin{equation}
\dot{x}(t) = A x(t) + B u(t), 
\qquad
x(0)=x_0,
\qquad
t\in[0,T].
\end{equation}

For Example 1, the data are
\begin{equation}
A=
\begin{pmatrix}
0 & 1\\
0 & 0
\end{pmatrix},
\qquad
B=
\begin{pmatrix}
0\\
1
\end{pmatrix},
\qquad
Q=I_2,
\qquad
Q_T=I_2,
\qquad
R=10^{-2},
\qquad
T=1,
\qquad
x_0=
\begin{pmatrix}
1\\
0
\end{pmatrix}.
\end{equation}

Equivalently, in components, the dynamics are
\begin{equation}
\dot{x}_1(t)=x_2(t),
\qquad
\dot{x}_2(t)=u(t),
\qquad
x_1(0)=1,
\qquad
x_2(0)=0.
\end{equation}

The cost functional becomes
\begin{equation}
J[u]
=
x_1(T)^2 + x_2(T)^2
+
\int_0^T
\left(
x_1(t)^2 + x_2(t)^2 + 10^{-2}u(t)^2
\right)\,dt.
\end{equation}

## Hamiltonian

The Hamiltonian is
\begin{equation}
H(p,x)
=
\min_{u}
\left(
p^\top (Ax+Bu) + x^\top Q x + u^\top R u
\right).
\end{equation}

Since the Hamiltonian is quadratic in $u$, the optimal control is obtained from the first-order condition
\begin{equation}
u^*(p) = -\frac{1}{2} R^{-1} B^\top p.
\end{equation}

Substituting this into the Hamiltonian gives
\begin{equation}
H(p,x)
=
p^\top A x + x^\top Q x
-
\frac{1}{4}\, p^\top B R^{-1} B^\top p.
\end{equation}

## Pontryagin maximum principle

The canonical system is
\begin{align}
\dot{x}(t) &= \nabla_p H(p(t),x(t)),\\
\dot{p}(t) &= -\nabla_x H(p(t),x(t)),
\end{align}
with boundary conditions
\begin{equation}
x(0)=x_0,
\qquad
p(T)=\nabla g(x(T)) = 2 Q_T x(T),
\end{equation}
where $g(x)=x^\top Q_T x$.

For this LQR problem, the PMP system becomes
\begin{align}
\dot{x}(t) &= A x(t) - \frac{1}{2} B R^{-1} B^\top p(t),\\
\dot{p}(t) &= -A^\top p(t) - 2Q x(t),
\end{align}
with
\begin{equation}
x(0)=x_0,
\qquad
p(T)=2Q_T x(T).
\end{equation}

## Riccati formulation

To obtain an independent benchmark solution, we introduce the ansatz
\begin{equation}
p(t) = 2 P(t) x(t),
\end{equation}
where $P(t)\in\mathbb{R}^{2\times 2}$ is a matrix-valued function.

Since $p(T)=2Q_T x(T)$, this implies the terminal condition
\begin{equation}
P(T)=Q_T.
\end{equation}

Substituting $p(t)=2P(t)x(t)$ into the PMP system yields the Riccati differential equation
\begin{equation}
-\dot{P}(t)
=
A^\top P(t) + P(t)A - P(t) B R^{-1} B^\top P(t) + Q,
\qquad
P(T)=Q_T.
\end{equation}

Once $P(t)$ is known, the optimal feedback control is
\begin{equation}
u^*(t) = - R^{-1} B^\top P(t) x(t),
\end{equation}
and the closed-loop state equation becomes
\begin{equation}
\dot{x}(t)
=
\left( A - B R^{-1} B^\top P(t) \right)x(t),
\qquad
x(0)=x_0.
\end{equation}

Therefore, the Riccati benchmark procedure is:

1. Solve the matrix Riccati equation backward in time for $P(t)$.
2. Solve the closed-loop state equation forward in time for $x(t)$.
3. Recover the control $u(t)$ and costate $p(t)=2P(t)x(t)$.
4. Compute the objective value $J[u]$.

## Step 1: Define the problem data and the Riccati ODE

The first implementation step is to define the data of the LQR problem and the right-hand side of the Riccati differential equation.

At this stage, we use two functions:

- `build_lqr_example_1()`
- `riccati_rhs(t, P_flat, A, B, Q, R_inv)`

<details>
    
### Function `build_lqr_example_1()`

This function collects all the data of Example 1 in a single dictionary:
- the matrices $A$, $B$, $Q$, $Q_T$, $R$,
- the inverse $R^{-1}$,
- the final time $T$,
- the initial condition $x_0$.

This is useful because all later routines use the same problem data, and storing everything in one dictionary makes the script cleaner and more modular.

### Function `riccati_rhs(t, P_flat, A, B, Q, R_inv)`

This function implements the Riccati differential equation
\begin{equation}
-\dot{P}(t)
=
A^\top P(t) + P(t)A - P(t)BR^{-1}B^\top P(t) + Q,
\qquad
P(T)=Q_T.
\end{equation}

Equivalently, for numerical integration we write
\begin{equation}
\dot{P}(t)
=
- A^\top P(t) - P(t)A + P(t)BR^{-1}B^\top P(t) - Q.
\end{equation}

Since `solve_ivp` works with one-dimensional arrays, the matrix $P(t)$ is stored in flattened form. Therefore:

- `P_flat` is the flattened version of $P(t)$,
- inside the function it is reshaped into a matrix,
- after computing $\dot{P}(t)$, the result is flattened again before returning it.

This function does not solve the Riccati equation yet; it only provides the ODE right-hand side that will be passed later to the numerical solver.
</details>

In [8]:
show_full_function(build_lqr_example_1)

In [9]:
show_full_function(riccati_rhs)

## Step 2: Solve the Riccati equation and verify its basic properties

Once the problem data and the Riccati ODE right-hand side are defined, the next step is to solve the matrix differential equation numerically and check that the solution has the expected structure.

At this stage, we use two functions:

- `solve_riccati_equation(data, num_eval_points=1000, rtol=1e-10, atol=1e-12)`
- `riccati_diagnostics(data, riccati_result)`

<details>

### Function `solve_riccati_equation(...)`

This function solves the Riccati differential equation
\begin{equation}
-\dot{P}(t)
=
A^\top P(t) + P(t)A - P(t)BR^{-1}B^\top P(t) + Q,
\qquad
P(T)=Q_T,
\end{equation}
by integrating backward in time from $t=T$ to $t=0$.

Numerically, the equation is written in forward form as
\begin{equation}
\dot{P}(t)
=
- A^\top P(t) - P(t)A + P(t)BR^{-1}B^\top P(t) - Q.
\end{equation}

The solver returns:

- a descending time grid from $T$ to $0$,
- the corresponding matrices $P(t)$ on that grid,
- the raw `solve_ivp` solution object,
- a callable function `P_of_t(t)` that evaluates the numerical Riccati solution at any $t\in[0,T]$,
- the terminal condition error $\|P(T)-Q_T\|$.

The role of this function is therefore to compute the benchmark Riccati solution that will later be used to generate the state, control, and costate trajectories.

### Function `riccati_diagnostics(...)`

After solving the Riccati equation, we perform a first validation of the numerical solution.

This function computes three basic diagnostic quantities:

1. the terminal condition error
\begin{equation}
\|P(T)-Q_T\|,
\end{equation}

2. the maximum symmetry defect along the stored time grid
\begin{equation}
\max_t \|P(t)-P(t)^\top\|,
\end{equation}

3. the initial Riccati matrix $P(0)$.

These checks are important because, for an LQR problem with symmetric matrices $Q$ and $Q_T$, the Riccati solution should remain symmetric up to numerical error. Thus, the symmetry defect provides a simple consistency check of the numerical integration.
    
</details>

In [10]:
show_full_function(solve_riccati_equation)

In [11]:
show_full_function(riccati_diagnostics)

## Step 3: Solve the closed-loop state equation and recover the trajectories

Once the Riccati solution $P(t)$ is available, the next step is to solve the state equation under the optimal feedback law.

At this stage, we use two functions:

- `state_rhs(t, x, A, B, R_inv, P_of_t)`
- `solve_state_equation(data, riccati_result, num_eval_points=1000, rtol=1e-10, atol=1e-12)`

### Function `state_rhs(...)`

This function implements the closed-loop state equation
\begin{equation}
\dot{x}(t)
=
\left(A - B R^{-1} B^\top P(t)\right)x(t),
\qquad
x(0)=x_0.
\end{equation}

Here, the matrix $P(t)$ is not recomputed inside the function. Instead, it is evaluated through the callable `P_of_t(t)` obtained from the Riccati solver.

Thus, for each time $t$, the function performs the following steps:

1. evaluate the Riccati matrix $P(t)$,
2. build the closed-loop matrix
   \begin{equation}
   A_{\mathrm{cl}}(t)=A-BR^{-1}B^\top P(t),
   \end{equation}
3. return the vector field
   \begin{equation}
   \dot{x}(t)=A_{\mathrm{cl}}(t)x(t).
   \end{equation}

Therefore, `state_rhs(...)` provides the right-hand side needed by the ODE solver for the state dynamics.

### Function `solve_state_equation(...)`

This function solves the closed-loop state equation forward in time, from $t=0$ to $t=T$, using the initial condition $x(0)=x_0$.

Besides the state trajectory, this function also reconstructs the optimal control and the costate from the Riccati solution:

\begin{equation}
u(t) = -R^{-1} B^\top P(t)x(t),
\end{equation}

\begin{equation}
p(t)=2P(t)x(t).
\end{equation}

The solver returns:

- an ascending time grid from $0$ to $T$,
- the state trajectory $x(t)$ on that grid,
- the control trajectory $u(t)$ on that grid,
- the costate trajectory $p(t)$ on that grid,
- the raw `solve_ivp` solution object,
- the initial condition error $\|x(0)-x_0\|$.

As discussed before, the quantity $\|x(0)-x_0\|$ is only a sanity check, since the initial condition is imposed directly by the numerical integration.

### Why this step is important

At this stage, the Riccati matrix $P(t)$ has already encoded the solution of the LQR problem. Solving the state equation allows us to recover the full optimal trajectory and then reconstruct the corresponding control and costate.

So this is the step where the benchmark becomes a complete reference solution:
- $P(t)$ comes from the backward Riccati solve,
- $x(t)$ comes from the forward closed-loop solve,
- $u(t)$ and $p(t)$ are recovered from $P(t)$ and $x(t)$.

### Summary of this step

In this third step, we move from the matrix Riccati solution to the actual optimal trajectories:

1. define the closed-loop state dynamics using the computed $P(t)$,
2. solve the state equation forward in time,
3. reconstruct the control $u(t)$ and the costate $p(t)$.